# 04 — Klasifikasi Gambar Sederhana (PyTorch + MNIST)

Tujuan:
- Memuat dataset MNIST (angka tulisan tangan 28x28)
- Melatih CNN kecil di CPU/GPU
- Mengevaluasi akurasi test
- Menyimpan model terlatih (`labs/04-pytorch-cnn.pt`)

Catatan: GPU opsional. Untuk hasil cepat, epoch dibuat kecil.


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1) Dataset & DataLoader (MNIST)

In [ ]:
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tf)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=tf)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=2)
len(train_ds), len(test_ds)

## 2) Model CNN sederhana

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)  # 1x28x28 -> 16x28x28
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1) # 16x14x14 -> 32x14x14
        self.pool  = nn.MaxPool2d(2,2)             # downsample 2x
        self.fc1   = nn.Linear(32*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x))) # 1x28x28 -> 16x14x14
        x = self.pool(torch.relu(self.conv2(x))) # 16x14x14 -> 32x7x7
        x = x.view(x.size(0), -1)                # flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

## 3) Training & Evaluasi
Untuk demo cepat, jalankan 2 epoch. Naikkan epoch untuk akurasi lebih tinggi.

In [ ]:
def train_one_epoch():
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        pred = logits.argmax(1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate():
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss_sum += loss.item() * xb.size(0)
        pred = logits.argmax(1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return loss_sum/total, correct/total

EPOCHS = 2
for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_one_epoch()
    te_loss, te_acc = evaluate()
    print(f"Epoch {epoch}: train {tr_loss:.4f}/{tr_acc:.4f} | test {te_loss:.4f}/{te_acc:.4f}")

## 4) Simpan & Muat Ulang Model

In [ ]:
out_path = Path("ai-engineering-roadmap/labs/04-pytorch-cnn.pt")
torch.save(model.state_dict(), out_path)
out_path

Muat ulang (opsional):

In [ ]:
loaded = SimpleCNN().to(device)
loaded.load_state_dict(torch.load(out_path, map_location=device))
loaded.eval()
print("Loaded OK")

Selesai.

Lanjutkan ke tugas NLP (labs/05-nlp-sentiment-transformers.ipynb) atau ke deployment (docs/11-deploy-fastapi-docker.md).